# LeetCode 951 — Flip Equivalent Binary Trees bằng Python

Link đề: https://leetcode.com/problems/flip-equivalent-binary-trees/ · Độ khó: **Medium** · Chủ đề: Tree, Depth-First Search, Binary Tree

Skeleton LeetCode:

```python
# Definition for a binary tree node.
# class TreeNode:
#     def __init__(self, val=0, left=None, right=None):
#         self.val = val
#         self.left = left
#         self.right = right
class Solution:
    def flipEquiv(self, root1: Optional[TreeNode], root2: Optional[TreeNode]) -> bool:
        ...
```

Ghi chú: bài trả về `bool` (đáp án duy nhất). Vì LeetCode truyền vào `TreeNode`, notebook **tự dựng** `TreeNode`, hàm dựng cây từ danh sách kiểu LeetCode (`build_tree`, duyệt theo tầng, `None` là nút trống) và hàm in ngược (`to_level`). Oracle độc lập: **đúng nghĩa định nghĩa** — từ cây `X`, áp dụng liên tiếp phép lật tại một nút bất kỳ (BFS trên tập các cây đạt được) và kiểm tra có đến được `Y` hay không.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu phép **lật (flip)**: chọn một nút bất kỳ, đổi chỗ hai cây con trái/phải của nó; `X` là *flip equivalent* với `Y` nếu sau một số phép lật `X` trở thành `Y`.
2. Nắm công cụ: `TreeNode`, dựng cây từ danh sách kiểu LeetCode, đệ quy hai nhánh (`and`/`or`), dạng chuẩn tắc (canonical form), duyệt lặp bằng stack các cặp nút, ánh xạ cha → tập con.
3. Đi từ **liệt kê mọi cách lật** → **đệ quy so sánh hai trường hợp (thẳng / chéo)** → **dạng chuẩn tắc (sắp xếp hai con)** → **duyệt lặp ghép cặp theo giá trị** → **so sánh quan hệ cha–con**.
4. Hiểu mấu chốt: tương đương lật ⇔ cùng giá trị gốc và (con trái~con trái, phải~phải) **hoặc** (trái~phải, phải~trái); nhờ giá trị **duy nhất** nên chỉ có tối đa một trong hai khả năng khớp ở mỗi nút ⇒ `O(n)`.
5. Tránh lỗi kinh điển: chỉ so sánh thẳng (quên lật), dùng `and` thay `or`, quên so giá trị gốc, trả `True` khi một cây rỗng, truy cập `.val` của `None`, so sánh theo duyệt (inorder/tập giá trị) thay vì cấu trúc.

## 2. Đề bài & ví dụ

Với cây nhị phân `T`, **phép lật** là: chọn một nút bất kỳ và đổi chỗ cây con trái với cây con phải của nút đó.

Cây `X` **tương đương lật** với cây `Y` nếu và chỉ nếu ta có thể biến `X` thành `Y` sau một số phép lật.

Cho hai gốc `root1`, `root2`; trả về `true` nếu hai cây tương đương lật, ngược lại `false`.

**Ví dụ 1:** `root1 = [1,2,3,4,5,6,null,null,null,7,8]`, `root2 = [1,3,2,null,6,4,5,null,null,null,null,8,7]` → `true` (lật tại các nút `1`, `3`, `5`).

**Ví dụ 2:** `root1 = []`, `root2 = []` → `true`.

**Ví dụ 3:** `root1 = []`, `root2 = [1]` → `false`.

**Ràng buộc:** số nút mỗi cây trong `[0, 100]`; mỗi cây có các giá trị nút **duy nhất** trong `[0, 99]`.

## 3. Kiến thức nền

### 3.1 `TreeNode` và cây nhị phân

LeetCode cung cấp sẵn `TreeNode`; khi chạy ở máy ta phải tự định nghĩa. Mỗi nút có `val`, `left`, `right`; cây rỗng là `None`.

In [1]:
import itertools
import random
import time
from collections import deque
from functools import lru_cache
from typing import List, Optional


class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right

    def __repr__(self):
        return f"TreeNode({self.val})"


a = TreeNode(1, TreeNode(2), TreeNode(3))
print(a.val, a.left.val, a.right.val, a.left.left)

1 2 3 None


### 3.2 Dựng cây theo kiểu LeetCode (duyệt theo tầng, `None` là nút trống)

Danh sách `[1,2,3,4,5,6,null,...]` được đọc bằng hàng đợi: lấy lần lượt từng nút đã dựng, gán hai phần tử kế tiếp làm con trái, con phải (`None` nghĩa là không có con — và **các con của nút trống không xuất hiện** trong danh sách). `to_level` làm ngược lại, và bỏ các `None` ở cuối như LeetCode.

In [2]:
def build_tree(values):
    """Dựng cây từ danh sách theo tầng kiểu LeetCode (None = không có nút)."""
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0])
    queue = deque([root])
    i = 1
    while queue and i < len(values):
        node = queue.popleft()
        if i < len(values) and values[i] is not None:
            node.left = TreeNode(values[i])
            queue.append(node.left)
        i += 1
        if i < len(values) and values[i] is not None:
            node.right = TreeNode(values[i])
            queue.append(node.right)
        i += 1
    return root


def to_level(root):
    """Ngược lại của build_tree: danh sách theo tầng, bỏ None ở cuối."""
    if root is None:
        return []
    out, queue = [], deque([root])
    while queue:
        node = queue.popleft()
        if node is None:
            out.append(None)
            continue
        out.append(node.val)
        queue.append(node.left)
        queue.append(node.right)
    while out and out[-1] is None:
        out.pop()
    return out


def tup(root):
    """Dạng bộ lồng nhau (val, trái, phải); None = cây rỗng. Dùng để so sánh/băm cây."""
    return None if root is None else (root.val, tup(root.left), tup(root.right))


r1 = build_tree([1, 2, 3, 4, 5, 6, None, None, None, 7, 8])
r2 = build_tree([1, 3, 2, None, 6, 4, 5, None, None, None, None, 8, 7])
print(tup(r1))
print(tup(r2))
assert to_level(r1) == [1, 2, 3, 4, 5, 6, None, None, None, 7, 8]
assert to_level(r2) == [1, 3, 2, None, 6, 4, 5, None, None, None, None, 8, 7]
assert build_tree([]) is None and to_level(None) == []

(1, (2, (4, None, None), (5, (7, None, None), (8, None, None))), (3, (6, None, None), None))
(1, (3, None, (6, None, None)), (2, (4, None, None), (5, (8, None, None), (7, None, None))))


### 3.3 Phép lật theo đúng định nghĩa

Biểu diễn cây bằng bộ `(val, trái, phải)`; một phép lật tại một nút là hoán đổi hai thành phần cuối của bộ ở nút đó. `one_flips(t)` sinh mọi cây thu được sau **đúng một** phép lật (tại từng nút). Bao đóng bắc cầu cho ta tập cây đạt được (`reachable`) — đây là định nghĩa nguyên văn của đề.

In [3]:
def one_flips(t):
    if t is None:
        return
    v, l, r = t
    yield (v, r, l)                       # lật tại chính nút này
    for l2 in one_flips(l):
        yield (v, l2, r)                  # lật ở đâu đó trong cây con trái
    for r2 in one_flips(r):
        yield (v, l, r2)                  # lật ở đâu đó trong cây con phải


@lru_cache(maxsize=None)
def reachable(t):
    """Mọi cây đạt được từ t sau một số (>= 0) phép lật: BFS theo định nghĩa."""
    seen = {t}
    queue = deque([t])
    while queue:
        cur = queue.popleft()
        for nxt in one_flips(cur):
            if nxt not in seen:
                seen.add(nxt)
                queue.append(nxt)
    return frozenset(seen)


x = tup(build_tree([1, 2, 3]))
print(x, "->", sorted(reachable(x), key=str))

(1, (2, None, None), (3, None, None)) -> [(1, (2, None, None), (3, None, None)), (1, (3, None, None), (2, None, None))]


### 3.4 Chữ ký giá trị duy nhất

Ràng buộc *giá trị duy nhất trong mỗi cây* quan trọng: nó cho phép biết nút nào của `Y` tương ứng với nút nào của `X` chỉ bằng giá trị. Nếu trùng giá trị, các cách giải dựa vào giá trị (ghép cặp, ánh xạ cha→con) có thể sai — ta sẽ minh hoạ ở mục 15.

## 4. Ý tưởng nền tảng / trực giác

Hai cây chỉ khác nhau bởi các phép lật ⇒ chúng "giống nhau nếu coi hai con của mỗi nút là một **cặp không thứ tự**". Nhìn vào gốc:

- gốc phải có **cùng giá trị** (phép lật không đổi giá trị);
- hai cây con của `root1` phải ghép được với hai cây con của `root2`, hoặc theo chiều **thẳng** (trái–trái, phải–phải) hoặc theo chiều **chéo** (trái–phải, phải–trái) — chiều chéo chính là lật tại gốc;
- mỗi cặp cây con lại là bài toán cùng dạng ⇒ đệ quy.

Ví dụ 1: gốc `1` và `1` khớp. Thẳng: `2` vs `3` lệch ⇒ thử chéo: `root1.left = 2` với `root2.right = 2` ✓; `root1.right = 3` với `root2.left = 3` ✓ ⇒ lật tại gốc. Tiếp tục xuống dưới: tại `3`, con là `(6, None)` còn bên kia `(None, 6)` ⇒ lật; tại `5`, `(7, 8)` vs `(8, 7)` ⇒ lật; còn lại khớp thẳng.

In [4]:
r1 = build_tree([1, 2, 3, 4, 5, 6, None, None, None, 7, 8])
r2 = build_tree([1, 3, 2, None, 6, 4, 5, None, None, None, None, 8, 7])
print("root1: gốc", r1.val, "| con:", r1.left.val, r1.right.val, "| con của 2:", r1.left.left.val, r1.left.right.val)
print("root2: gốc", r2.val, "| con:", r2.left.val, r2.right.val, "| con của 2:", r2.right.left.val, r2.right.right.val)
print("5 của root1 có con:", r1.left.right.left.val, r1.left.right.right.val, "| 5 của root2 có con:", r2.right.right.left.val, r2.right.right.right.val)

root1: gốc 1 | con: 2 3 | con của 2: 4 5
root2: gốc 1 | con: 3 2 | con của 2: 4 5
5 của root1 có con: 7 8 | 5 của root2 có con: 8 7


## 5. Brute force — liệt kê mọi cách lật

**Ý tưởng:** tập mọi cây có thể thu được từ `X` bằng cách quyết định, **ở mỗi nút**, lật hay không. Với bộ lồng nhau: `variants(None) = {None}`; `variants((v,l,r)) = {(v,a,b), (v,b,a) | a ∈ variants(l), b ∈ variants(r)}`. Sau đó kiểm tra `tup(Y)` có thuộc tập đó không. Số cây tối đa `2^(số nút trong)` — chỉ dùng cho cây nhỏ; đây cũng là một **oracle** (cùng định nghĩa nhưng cài đặt khác với `reachable`; ta sẽ chứng minh hai tập này bằng nhau trên mọi cây nhỏ ở mục 12).

In [5]:
def variants(t):
    if t is None:
        return {None}
    v, l, r = t
    out = set()
    for a in variants(l):
        for b in variants(r):
            out.add((v, a, b))                # không lật tại nút này
            out.add((v, b, a))                # lật tại nút này
    return out


def flip_equiv_brute(root1, root2):
    return tup(root2) in variants(tup(root1))


print(flip_equiv_brute(build_tree([1, 2, 3]), build_tree([1, 3, 2])), flip_equiv_brute(build_tree([1, 2, 3]), build_tree([1, 2, 4])))
print(len(variants(tup(build_tree([1, 2, 3, 4, 5])))), "cây thu được từ [1,2,3,4,5] (nút 1 và 2 có con: 2^2 = 4)")

True False
4 cây thu được từ [1,2,3,4,5] (nút 1 và 2 có con: 2^2 = 4)


In [6]:
# Code nộp LeetCode cho Brute force (liệt kê mọi cách lật; sẽ chậm khi nhiều nút)
class SolutionBrute:
    def flipEquiv(self, root1: Optional[TreeNode], root2: Optional[TreeNode]) -> bool:
        def as_tuple(n):
            return None if n is None else (n.val, as_tuple(n.left), as_tuple(n.right))

        def variants(t):
            if t is None:
                return {None}
            v, l, r = t
            out = set()
            for a in variants(l):
                for b in variants(r):
                    out.add((v, a, b))
                    out.add((v, b, a))
            return out

        return as_tuple(root2) in variants(as_tuple(root1))

**Độ phức tạp:** `O(n · 2^k)` thời gian và bộ nhớ với `k` là số nút trong (có con) — mũ. Với `n = 100` hoàn toàn bất khả thi.

**Nút thắt:** ta dựng cả tập `2^k` cây chỉ để hỏi một câu hỏi "có khớp không". Nhưng việc khớp có cấu trúc đệ quy: ở mỗi nút chỉ có hai cách ghép con.

## 6. Bước đột phá

`X ~ Y` ⇔ cả hai rỗng, hoặc: gốc cùng giá trị **và** (`X.l ~ Y.l` và `X.r ~ Y.r`) **hoặc** (`X.l ~ Y.r` và `X.r ~ Y.l`). Đệ quy trực tiếp; với giá trị duy nhất, mỗi nút của `X` khớp nhiều nhất một nút của `Y` nên tổng việc là `O(n)`.

## 7. Các cách tiếp theo

Ngoài Brute force (mục 5), ta có thêm 4 cách: **Cách 2** đệ quy hai nhánh, **Cách 3** dạng chuẩn tắc (sắp xếp hai con), **Cách 4** duyệt lặp bằng stack các cặp nút, **Cách 5** so sánh ánh xạ cha → tập con.

### Cách 2 — Đệ quy hai nhánh (đáp án chính)

**Giải thích từng đoạn:**

1. Cả hai rỗng ⇒ `True`.
2. Một bên rỗng, hoặc giá trị gốc khác nhau ⇒ `False`.
3. Ngược lại: `True` nếu **thẳng** hoặc **chéo** khớp. Dùng `and` trong mỗi nhánh (cả hai con phải khớp) và `or` giữa hai nhánh.

In [7]:
def flip_equiv(root1, root2):
    if root1 is None and root2 is None:
        return True
    if root1 is None or root2 is None or root1.val != root2.val:
        return False
    straight = flip_equiv(root1.left, root2.left) and flip_equiv(root1.right, root2.right)
    if straight:
        return True
    return flip_equiv(root1.left, root2.right) and flip_equiv(root1.right, root2.left)


print(flip_equiv(r1, r2), flip_equiv(None, None), flip_equiv(None, build_tree([1])))
assert flip_equiv(r1, r2) is True and flip_equiv(None, None) is True and flip_equiv(None, build_tree([1])) is False

True True False


In [8]:
# Code nộp LeetCode cho Cách 2
class SolutionRecursion:
    def flipEquiv(self, root1: Optional[TreeNode], root2: Optional[TreeNode]) -> bool:
        if root1 is None and root2 is None:
            return True
        if root1 is None or root2 is None or root1.val != root2.val:
            return False
        return (self.flipEquiv(root1.left, root2.left) and self.flipEquiv(root1.right, root2.right)) or \
               (self.flipEquiv(root1.left, root2.right) and self.flipEquiv(root1.right, root2.left))

**Độ phức tạp:** với giá trị duy nhất, mỗi nút của `root1` chỉ có thể khớp **một** nút của `root2` (nút cùng giá trị); các lời gọi còn lại thất bại ngay ở kiểm tra giá trị. Số lời gọi `≤ 4n + 1` ⇒ `O(n)` thời gian; ngăn xếp đệ quy `O(h)`.

### Cách 3 — Dạng chuẩn tắc (canonical form)

**Ý tưởng:** biến mỗi cây thành một dạng **không phụ thuộc thứ tự con**: `canon(n) = (n.val, c1, c2)` trong đó `c1 ≤ c2` là hai dạng chuẩn tắc của hai con **sắp xếp** (cây rỗng là `()`). Hai cây tương đương lật ⇔ dạng chuẩn tắc bằng nhau. Cách này đúng cả khi có giá trị trùng.

In [9]:
def canon(node):
    if node is None:
        return ()
    c1, c2 = sorted([canon(node.left), canon(node.right)])     # thứ tự con không còn quan trọng
    return (node.val, c1, c2)


def flip_equiv_canon(root1, root2):
    return canon(root1) == canon(root2)


print(canon(build_tree([1, 2, 3])), "==", canon(build_tree([1, 3, 2])))
assert flip_equiv_canon(r1, r2) and not flip_equiv_canon(None, build_tree([1])) and flip_equiv_canon(None, None)

(1, (2, (), ()), (3, (), ())) == (1, (2, (), ()), (3, (), ()))


In [10]:
# Code nộp LeetCode cho Cách 3
class SolutionCanon:
    def flipEquiv(self, root1: Optional[TreeNode], root2: Optional[TreeNode]) -> bool:
        def canon(node):
            if node is None:
                return ()
            c1, c2 = sorted([canon(node.left), canon(node.right)])
            return (node.val, c1, c2)

        return canon(root1) == canon(root2)

**Độ phức tạp:** mỗi nút sắp xếp hai phần tử và so sánh các bộ lồng nhau: tổng `O(n)` bộ nhớ cho các dạng; so sánh/sắp xếp dạng con tốn `O(kích thước)` nên trường hợp xấu `O(n · h)` (≤ `O(n^2)`, với `n ≤ 100` rất nhỏ). Bộ nhớ `O(n)`.

### Cách 4 — Duyệt lặp bằng stack các cặp nút

**Ý tưởng (giá trị duy nhất):** giữ stack các cặp `(a, b)` cần khớp. Với cặp hợp lệ (cùng giá trị) ta **quyết định** ghép thẳng hay chéo bằng giá trị con trái: nếu `a.left` và `b.left` có cùng giá trị (hoặc cùng `None`) thì ghép thẳng; ngược lại ghép chéo. Các cặp sai sẽ bị phát hiện khi lấy ra.

In [11]:
def flip_equiv_iter(root1, root2):
    def lv(n):
        return None if n is None else n.val

    stack = [(root1, root2)]
    while stack:
        a, b = stack.pop()
        if a is None and b is None:
            continue
        if a is None or b is None or a.val != b.val:
            return False
        if lv(a.left) == lv(b.left):                 # trái khớp trái (kể cả cùng None) => ghép thẳng
            stack.append((a.left, b.left))
            stack.append((a.right, b.right))
        else:                                        # ngược lại phải lật => ghép chéo
            stack.append((a.left, b.right))
            stack.append((a.right, b.left))
    return True


assert flip_equiv_iter(r1, r2) and flip_equiv_iter(None, None) and not flip_equiv_iter(None, build_tree([1]))
print(flip_equiv_iter(r1, r2))

True


In [12]:
# Code nộp LeetCode cho Cách 4 (dựa trên giá trị duy nhất)
class SolutionIter:
    def flipEquiv(self, root1: Optional[TreeNode], root2: Optional[TreeNode]) -> bool:
        stack = [(root1, root2)]
        while stack:
            a, b = stack.pop()
            if a is None and b is None:
                continue
            if a is None or b is None or a.val != b.val:
                return False
            al = a.left.val if a.left else None
            bl = b.left.val if b.left else None
            if al == bl:
                stack.append((a.left, b.left))
                stack.append((a.right, b.right))
            else:
                stack.append((a.left, b.right))
                stack.append((a.right, b.left))
        return True

**Độ phức tạp:** mỗi nút được ghép đúng một lần ⇒ `O(n)` thời gian, `O(n)` bộ nhớ (stack). Không đệ quy nên không lo giới hạn độ sâu.

### Cách 5 — So sánh quan hệ cha → tập con

**Ý tưởng (giá trị duy nhất):** các phép lật chỉ đổi **bên** (trái/phải) của con, không đổi quan hệ cha–con. Với giá trị duy nhất, cây không thứ tự được xác định bởi gốc và ánh xạ `cha → {tập giá trị các con}`. Hai cây tương đương lật ⇔ cùng gốc và hai ánh xạ bằng nhau.

In [13]:
def parent_map(root):
    """(giá trị gốc, {giá trị nút: frozenset giá trị các con}); cây rỗng -> None."""
    if root is None:
        return None
    mp = {}
    stack = [root]
    while stack:
        node = stack.pop()
        kids = [c for c in (node.left, node.right) if c is not None]
        mp[node.val] = frozenset(c.val for c in kids)
        stack.extend(kids)
    return (root.val, mp)


def flip_equiv_map(root1, root2):
    return parent_map(root1) == parent_map(root2)


print(parent_map(build_tree([1, 2, 3, 4])))
assert flip_equiv_map(r1, r2) and flip_equiv_map(None, None) and not flip_equiv_map(None, build_tree([1]))

(1, {1: frozenset({2, 3}), 3: frozenset(), 2: frozenset({4}), 4: frozenset()})


In [14]:
# Code nộp LeetCode cho Cách 5 (dựa trên giá trị duy nhất)
class SolutionMap:
    def flipEquiv(self, root1: Optional[TreeNode], root2: Optional[TreeNode]) -> bool:
        def parent_map(root):
            if root is None:
                return None
            mp, stack = {}, [root]
            while stack:
                node = stack.pop()
                kids = [c for c in (node.left, node.right) if c is not None]
                mp[node.val] = frozenset(c.val for c in kids)
                stack.extend(kids)
            return (root.val, mp)

        return parent_map(root1) == parent_map(root2)

**Độ phức tạp:** `O(n)` thời gian, `O(n)` bộ nhớ.

## 8. TRACE từng bước

**Trace Cách 2** với ví dụ 1. Mỗi lời gọi `(a, b)` in giá trị hai nút và nhánh được chọn (thụt dòng theo độ sâu):

| cặp | so sánh | quyết định |
|---|---|---|
| `(1,1)` | thẳng: `2` vs `3` lệch | chéo: `(2,2)` và `(3,3)` |
| `(2,2)` | thẳng: `(4,4)`, `(5,5)` | khớp thẳng |
| `(5,5)` | thẳng: `7` vs `8` lệch | chéo: `(7,7)`, `(8,8)` |
| `(3,3)` | thẳng: `6` vs `None` lệch | chéo: `(6,6)`, `(None,None)` |

Vậy cần lật ở các nút `1`, `5`, `3`: đúng như đề. Hàm trace in từng lời gọi:

In [15]:
def trace_recursion(root1, root2):
    calls = [0]

    def rec(a, b, depth):
        calls[0] += 1
        pad = "  " * depth
        va = None if a is None else a.val
        vb = None if b is None else b.val
        if a is None and b is None:
            print(f"{pad}({va},{vb}) cả hai rỗng -> True")
            return True
        if a is None or b is None or a.val != b.val:
            print(f"{pad}({va},{vb}) khác nhau -> False")
            return False
        print(f"{pad}({va},{vb}) thử THẲNG")
        if rec(a.left, b.left, depth + 1) and rec(a.right, b.right, depth + 1):
            print(f"{pad}=> ({va},{vb}) khớp thẳng")
            return True
        print(f"{pad}({va},{vb}) thử CHÉO (lật tại {va})")
        ok = rec(a.left, b.right, depth + 1) and rec(a.right, b.left, depth + 1)
        print(f"{pad}=> ({va},{vb}) {'khớp chéo' if ok else 'KHÔNG khớp'}")
        return ok

    res = rec(root1, root2, 0)
    print("kết quả:", res, "| số lời gọi:", calls[0])
    return res


assert trace_recursion(r1, r2) is True

(1,1) thử THẲNG
  (2,3) khác nhau -> False
(1,1) thử CHÉO (lật tại 1)
  (2,2) thử THẲNG
    (4,4) thử THẲNG
      (None,None) cả hai rỗng -> True
      (None,None) cả hai rỗng -> True
    => (4,4) khớp thẳng
    (5,5) thử THẲNG
      (7,8) khác nhau -> False
    (5,5) thử CHÉO (lật tại 5)
      (7,7) thử THẲNG
        (None,None) cả hai rỗng -> True
        (None,None) cả hai rỗng -> True
      => (7,7) khớp thẳng
      (8,8) thử THẲNG
        (None,None) cả hai rỗng -> True
        (None,None) cả hai rỗng -> True
      => (8,8) khớp thẳng
    => (5,5) khớp chéo
  => (2,2) khớp thẳng
  (3,3) thử THẲNG
    (6,None) khác nhau -> False
  (3,3) thử CHÉO (lật tại 3)
    (6,6) thử THẲNG
      (None,None) cả hai rỗng -> True
      (None,None) cả hai rỗng -> True
    => (6,6) khớp thẳng
    (None,None) cả hai rỗng -> True
  => (3,3) khớp chéo
=> (1,1) khớp chéo
kết quả: True | số lời gọi: 20


Với giá trị duy nhất, các nhánh chéo thất bại **ngay** ở kiểm tra giá trị (1 lời gọi), nên số lời gọi nhỏ: đây là lý do `O(n)`.

**Trace Cách 3 (dạng chuẩn tắc)** với cây `[1,2,3]` và `[1,3,2]`: `canon(2) = (2, (), ())`, `canon(3) = (3, (), ())`; sắp xếp `[(2,(),()), (3,(),())]` cho `(1, (2,(),()), (3,(),()))` ở cả hai cây ⇒ bằng nhau.

In [16]:
def trace_canon(root):
    def rec(node, depth):
        pad = "  " * depth
        if node is None:
            return ()
        left, right = rec(node.left, depth + 1), rec(node.right, depth + 1)
        res = (node.val,) + tuple(sorted([left, right]))
        print(f"{pad}nút {node.val}: trái={left} phải={right} -> {res}")
        return res

    return rec(root, 0)


c1 = trace_canon(build_tree([1, 2, 3]))
c2 = trace_canon(build_tree([1, 3, 2]))
assert c1 == c2

  nút 2: trái=() phải=() -> (2, (), ())
  nút 3: trái=() phải=() -> (3, (), ())
nút 1: trái=(2, (), ()) phải=(3, (), ()) -> (1, (2, (), ()), (3, (), ()))
  nút 3: trái=() phải=() -> (3, (), ())
  nút 2: trái=() phải=() -> (2, (), ())
nút 1: trái=(3, (), ()) phải=(2, (), ()) -> (1, (2, (), ()), (3, (), ()))


## 9. Code nộp LeetCode

Chọn **Cách 2** (đệ quy hai nhánh): ngắn, đúng cả khi giá trị trùng (nếu đề đổi), không phụ thuộc giả thiết. Giữ nguyên `class Solution`, `flipEquiv(self, root1, root2)`:

In [17]:
class Solution:
    def flipEquiv(self, root1: Optional[TreeNode], root2: Optional[TreeNode]) -> bool:
        if root1 is None and root2 is None:                       # cả hai rỗng: giống nhau
            return True
        if root1 is None or root2 is None or root1.val != root2.val:   # lệch hình dạng hoặc giá trị gốc
            return False
        # thẳng (không lật tại gốc) hoặc chéo (lật tại gốc)
        return (self.flipEquiv(root1.left, root2.left) and self.flipEquiv(root1.right, root2.right)) or \
               (self.flipEquiv(root1.left, root2.right) and self.flipEquiv(root1.right, root2.left))


print(Solution().flipEquiv(r1, r2), Solution().flipEquiv(build_tree([]), build_tree([])), Solution().flipEquiv(build_tree([]), build_tree([1])))
assert Solution().flipEquiv(r1, r2) is True and Solution().flipEquiv(None, None) is True and Solution().flipEquiv(None, build_tree([1])) is False

True True False


## 10. Vì sao đúng

**(a) Đặc trưng đệ quy.** Gọi `F(X)` là tập cây thu được từ `X` bằng mọi dãy phép lật. Khẳng định: với `X = (v, L, R)`,

`F(X) = { (v, a, b) : a ∈ F(L), b ∈ F(R) } ∪ { (v, b, a) : a ∈ F(L), b ∈ F(R) }`.

*Chứng minh.* Một dãy phép lật chia thành các phép lật **tại gốc** và các phép lật **bên trong** cây con. Phép lật tại một nút ở trong `L` (hoặc `R`) chỉ tác động lên cây con đó, **giao hoán** với phép lật ở gốc (lật gốc chỉ đổi chỗ hai cây con, không động đến nội dung chúng). Hai lần lật tại gốc triệt tiêu nhau. Vậy kết quả của một dãy bất kỳ có dạng: (có/không lật ở gốc) + (một phần tử của `F(L)` và một của `F(R)`) — đúng tập trên. Ngược lại, mọi phần tử của tập trên đều đạt được (thực hiện các phép lật tương ứng trong từng cây con và, nếu cần, một phép lật ở gốc). ∎

**(b) Cách 1 và Cách 2.** `variants` chính là công thức (a) nên Cách 1 đúng. Cách 2 là phát biểu `Y ∈ F(X)` ⇔ `Y` rỗng khi `X` rỗng, hoặc gốc bằng nhau và `(Y.l ∈ F(X.l) ∧ Y.r ∈ F(X.r)) ∨ (Y.l ∈ F(X.r) ∧ Y.r ∈ F(X.l))`: đây là công thức (a) viết dưới dạng thành viên.

**(c) Số lời gọi `≤ 4n + 1` khi giá trị duy nhất.** Gọi "lời gọi thành công" là lời gọi `(a, b)` với hai nút khác rỗng cùng giá trị. Vì giá trị duy nhất trong mỗi cây, mỗi nút `a` của `root1` tham gia **tối đa một** lời gọi thành công (nút `b` cùng giá trị duy nhất). Mỗi lời gọi thành công sinh tối đa `4` lời gọi con; các lời gọi con không thành công kết thúc ngay (`O(1)`). Tổng `≤ 1 + 4n`.

**(d) Cách 3.** `canon(X)` là bất biến theo mọi phép lật (sắp xếp hai con không phụ thuộc thứ tự) và xác định `X` **sai khác phép lật**: hai cây cùng dạng chuẩn tắc có cùng gốc và cùng (đa) tập dạng chuẩn tắc của hai con ⇒ theo quy nạp ghép được con–con thẳng hoặc chéo; ngược lại, tương đương lật cho cùng dạng. Cách này đúng cả khi giá trị trùng.

**(e) Cách 4.** Giả sử cặp `(a, b)` cùng giá trị. Có hai khả năng ghép: thẳng hoặc chéo. Nếu `lv(a.left) == lv(b.left)`: nếu ghép thẳng hợp lệ thì chọn thẳng đúng. Nếu ghép chéo cũng hợp lệ thì phải `lv(a.left) == lv(b.right)`, suy ra `lv(b.left) == lv(b.right)`; với giá trị duy nhất điều này chỉ xảy ra khi cả hai là `None` (giá trị `None` có thể lặp, giá trị thật thì không) — khi đó thẳng và chéo như nhau. Nếu `lv(a.left) != lv(b.left)` thì ghép thẳng chắc chắn sai (cặp `(a.left, b.left)` bị loại ở bước kiểm tra giá trị hoặc rỗng-không-rỗng), nên chỉ còn chéo. Vậy ghép theo giá trị con trái là lựa chọn duy nhất có thể đúng.

**(f) Cách 5.** Phép lật chỉ đổi bên của các con, không đổi quan hệ cha–con. Do đó `X ~ Y` ⇒ cùng gốc và cùng `cha → tập con`. Ngược lại, cùng gốc và cùng ánh xạ (giá trị duy nhất nên nút được xác định bởi giá trị): tại mỗi nút ta xếp các con của `X` theo đúng bên mà `Y` đặt — hai con thì lật hoặc không, một con thì lật nếu khác bên; các quyết định ở các nút độc lập. ∎

**(g) Kiểm chứng bằng máy.** Với **mọi** cặp cây có ≤ 4 nút (nhãn từ `{0,1,2,3}`, `485` cây, `235 225` cặp) mọi cách khớp oracle `reachable` (BFS các phép lật theo đúng định nghĩa) — mục 12D.

## 11. Time / Space Complexity

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Liệt kê mọi cách lật | `O(n · 2^k)` | `O(n · 2^k)` | oracle, cây nhỏ |
| 2. Đệ quy hai nhánh | `O(n)` (giá trị duy nhất) | `O(h)` | **đáp án chính** |
| 3. Dạng chuẩn tắc | `O(n · h)` | `O(n)` | đúng cả khi giá trị trùng |
| 4. Duyệt lặp ghép cặp | `O(n)` | `O(n)` | không đệ quy |
| 5. Ánh xạ cha → tập con | `O(n)` | `O(n)` | cần giá trị duy nhất |

(`n` là số nút, `h` chiều cao, `k` số nút trong.) Đo thực tế với cây 100 nút (các cách có thể chạy):

In [18]:
def random_tree(rng, n, labels=None):
    """Cây ngẫu nhiên n nút, nhãn duy nhất lấy từ labels (mặc định 0..99)."""
    if n == 0:
        return None
    pool = list(range(100)) if labels is None else list(labels)
    vals = rng.sample(pool, n)
    it = iter(vals)

    def make(m):
        if m == 0:
            return None
        node = TreeNode(next(it))
        k = rng.randint(0, m - 1)
        node.left = make(k)
        node.right = make(m - 1 - k)
        return node

    return make(n)


def copy_flip(root, rng, p=0.5):
    """Bản sao của cây với mỗi nút được lật ngẫu nhiên (xác suất p) — đúng định nghĩa nên LUÔN tương đương lật."""
    if root is None:
        return None
    l, r = copy_flip(root.left, rng, p), copy_flip(root.right, rng, p)
    if rng.random() < p:
        l, r = r, l
    return TreeNode(root.val, l, r)


def size(root):
    return 0 if root is None else 1 + size(root.left) + size(root.right)


def bench(fn, a, b, repeat=200):
    best = 1e9
    for _ in range(3):
        t0 = time.perf_counter()
        for _ in range(repeat):
            fn(a, b)
        best = min(best, (time.perf_counter() - t0) / repeat)
    return best


rng = random.Random(951)
ta = random_tree(rng, 100)
tb = copy_flip(ta, rng)
assert size(ta) == 100 and flip_equiv(ta, tb)
for name, fn in [("recursion", flip_equiv), ("canon", flip_equiv_canon), ("iter", flip_equiv_iter), ("map", flip_equiv_map)]:
    print(f"{name:9s}: {bench(fn, ta, tb) * 1e6:8.1f} us (n = 100)")

recursion:     20.8 us (n = 100)
canon    :     80.3 us (n = 100)
iter     :     36.1 us (n = 100)


map      :    158.6 us (n = 100)


## 12. Bộ test

Chia 6 nhóm: (A) ví dụ đề, (B) edge case, (C) phản ví dụ, (D) vét cạn **mọi cặp cây ≤ 4 nút** với oracle `reachable` (BFS các phép lật), (E) tính chất / ngẫu nhiên (cây tới 100 nút, seed cố định), (F) giới hạn.

### A. Ví dụ của đề

Mọi cách được thử trên cây dựng từ danh sách kiểu LeetCode; oracle ở đây là `reachable` (định nghĩa nguyên văn).

In [19]:
IMPLS = {
    "recursion": flip_equiv,
    "canon": flip_equiv_canon,
    "iter": flip_equiv_iter,
    "map": flip_equiv_map,
    "brute": flip_equiv_brute,
    "Solution": lambda a, b: Solution().flipEquiv(a, b),
    "SolutionRecursion": lambda a, b: SolutionRecursion().flipEquiv(a, b),
    "SolutionCanon": lambda a, b: SolutionCanon().flipEquiv(a, b),
    "SolutionIter": lambda a, b: SolutionIter().flipEquiv(a, b),
    "SolutionMap": lambda a, b: SolutionMap().flipEquiv(a, b),
    "SolutionBrute": lambda a, b: SolutionBrute().flipEquiv(a, b),
}


def oracle(root1, root2):
    """Định nghĩa nguyên văn: tup(root2) có nằm trong tập cây đạt được từ root1 bằng các phép lật không."""
    return tup(root2) in reachable(tup(root1))


def check(l1, l2, want=None, names=None):
    a, b = build_tree(l1), build_tree(l2)
    truth = oracle(a, b) if size(a) <= 8 else None
    if want is None:
        want = truth
    elif truth is not None:
        assert truth == want, (l1, l2, truth, want)
    for name in (names or IMPLS):
        got = IMPLS[name](build_tree(l1), build_tree(l2))
        assert got is want or got == want, (name, l1, l2, got, want)


ex = [
    ([1, 2, 3, 4, 5, 6, None, None, None, 7, 8], [1, 3, 2, None, 6, 4, 5, None, None, None, None, 8, 7], True),
    ([], [], True),
    ([], [1], False),
]
for l1, l2, want in ex:
    a, b = build_tree(l1), build_tree(l2)
    if size(a) <= 8:
        assert oracle(a, b) == want
    for name in IMPLS:
        if name in ("brute", "SolutionBrute") and size(a) > 8:
            continue                                     # liệt kê mọi cách lật: chỉ cho cây nhỏ
        assert IMPLS[name](build_tree(l1), build_tree(l2)) == want, (name, l1, l2)
print("ví dụ đề: các cách đúng (brute chỉ cho cây nhỏ)")

ví dụ đề: các cách đúng (brute chỉ cho cây nhỏ)


### B. Edge case

- Hai cây rỗng; một rỗng một không rỗng (cả hai chiều).
- Một nút: cùng giá trị / khác giá trị (kể cả `0`, giá trị nhỏ nhất).
- Cùng hình dạng nhưng khác giá trị ở lá.
- **Chuỗi trái vs chuỗi phải** (mỗi nút chỉ một con): tương đương lật.
- Hình dạng khác nhau (số nút khác nhau).
- Cây đối xứng (lật tại một nút cho cùng cây).
- Gốc khác nhau nhưng cùng tập giá trị.

In [20]:
edge = [
    ([], [], True), ([], [1], False), ([1], [], False), ([0], [0], True), ([0], [1], False),
    ([1, 2, 3], [1, 3, 2], True), ([1, 2, 3], [1, 2, 4], False), ([1, 2, 3], [1, 2, 3], True),
    ([1, 2], [1, None, 2], True), ([1, None, 2], [1, 2], True), ([1, 2, None, 3], [1, None, 2, None, 3], True),
    ([1, 2, None, 3], [1, None, 2, 3], True), ([1, 2, 3], [1, 2], False), ([1, 2], [2, 1], False),
    ([1, 2, 3, 4], [1, 3, 2, None, None, 4], True), ([1, 2, 3, 4], [1, 3, 2, None, None, None, 4], True),
    ([1, 2, 3, 4], [1, 3, 2, 4], False),
]
for l1, l2, want in edge:
    check(l1, l2, want)
print(len(edge), "edge case: mọi cách khớp oracle (định nghĩa nguyên văn)")

17 edge case: mọi cách khớp oracle (định nghĩa nguyên văn)


### C. Phản ví dụ: cài đặt sai phải bị bắt

In [21]:
def bug_no_flip(a, b):                              # chỉ so sánh thẳng
    if a is None and b is None:
        return True
    if a is None or b is None or a.val != b.val:
        return False
    return bug_no_flip(a.left, b.left) and bug_no_flip(a.right, b.right)


def bug_and_instead_of_or(a, b):                    # thẳng VÀ chéo đều phải đúng
    if a is None and b is None:
        return True
    if a is None or b is None or a.val != b.val:
        return False
    return (bug_and_instead_of_or(a.left, b.left) and bug_and_instead_of_or(a.right, b.right)) and \
           (bug_and_instead_of_or(a.left, b.right) and bug_and_instead_of_or(a.right, b.left))


def bug_none_is_true(a, b):                         # một bên rỗng thì trả True
    if a is None or b is None:
        return True
    if a.val != b.val:
        return False
    return (bug_none_is_true(a.left, b.left) and bug_none_is_true(a.right, b.right)) or \
           (bug_none_is_true(a.left, b.right) and bug_none_is_true(a.right, b.left))


def bug_no_root_check(a, b):                        # quên so giá trị gốc
    if a is None and b is None:
        return True
    if a is None or b is None:
        return False
    return (bug_no_root_check(a.left, b.left) and bug_no_root_check(a.right, b.right)) or \
           (bug_no_root_check(a.left, b.right) and bug_no_root_check(a.right, b.left))


def bug_value_multiset(a, b):                       # chỉ so tập giá trị, bỏ qua cấu trúc
    def vals(n):
        return [] if n is None else vals(n.left) + [n.val] + vals(n.right)
    return sorted(vals(a)) == sorted(vals(b))


def bug_root_only_flip(a, b):                       # chỉ cho lật tại gốc, các cây con phải giống hệt
    return tup(a) == tup(b) or (a is not None and b is not None and a.val == b.val and
                                 tup(a.left) == tup(b.right) and tup(a.right) == tup(b.left))


def bug_children_values_only(a, b):                 # chỉ so tập giá trị hai con của gốc (nông)
    if a is None or b is None:
        return a is b
    ca = {None if a.left is None else a.left.val, None if a.right is None else a.right.val}
    cb = {None if b.left is None else b.left.val, None if b.right is None else b.right.val}
    return a.val == b.val and ca == cb


def bug_attribute_error(a, b):                      # truy cập .val của con mà không kiểm tra None (lỗi chạy)
    if a is None and b is None:
        return True
    if a is None or b is None or a.val != b.val:
        return False
    if a.left.val == b.left.val:                   # AttributeError khi a.left là None
        return bug_attribute_error(a.left, b.left) and bug_attribute_error(a.right, b.right)
    return bug_attribute_error(a.left, b.right) and bug_attribute_error(a.right, b.left)


BUGS = [bug_no_flip, bug_and_instead_of_or, bug_none_is_true, bug_no_root_check, bug_value_multiset, bug_root_only_flip,
        bug_children_values_only, bug_attribute_error]
bug_cases = [
    ([1, 2, 3], [1, 3, 2]), ([], [1]), ([1], [2]), ([1, 2, 3], [1, 2, 4]), ([1, 2, 3], [1, 2]),
    ([1, 2, 3, 4, 5, 6, None, None, None, 7, 8], [1, 3, 2, None, 6, 4, 5, None, None, None, None, 8, 7]),
    ([1, 2, None, 3], [1, 2, None, None, 3]), ([1, 2], [1, None, 2]), ([1, 2, 3, 4], [1, 2, 3, None, None, 4]),
]
for bug in BUGS:
    found = None
    for l1, l2 in bug_cases:
        a, b = build_tree(l1), build_tree(l2)
        try:
            got = bug(a, b)
        except AttributeError:
            got = "AttributeError"
        if got != flip_equiv(a, b):
            found = (l1, l2, got)
            break
    assert found, bug.__name__
    print(f"{bug.__name__:26s} sai trên {found[0]} vs {found[1]}: ra {found[2]}, đúng {flip_equiv(build_tree(found[0]), build_tree(found[1]))}")

bug_no_flip                sai trên [1, 2, 3] vs [1, 3, 2]: ra False, đúng True
bug_and_instead_of_or      sai trên [1, 2, 3] vs [1, 3, 2]: ra False, đúng True
bug_none_is_true           sai trên [] vs [1]: ra True, đúng False
bug_no_root_check          sai trên [1] vs [2]: ra True, đúng False
bug_value_multiset         sai trên [1, 2, 3, 4] vs [1, 2, 3, None, None, 4]: ra True, đúng False
bug_root_only_flip         sai trên [1, 2, 3, 4, 5, 6, None, None, None, 7, 8] vs [1, 3, 2, None, 6, 4, 5, None, None, None, None, 8, 7]: ra False, đúng True
bug_children_values_only   sai trên [1, 2, 3, 4] vs [1, 2, 3, None, None, 4]: ra True, đúng False
bug_attribute_error        sai trên [1, 2, 3] vs [1, 3, 2]: ra AttributeError, đúng True


### D. Vét cạn **mọi cặp cây ≤ 4 nút**

`all_trees(labels)` sinh mọi cây dùng **đúng** tập nhãn `labels` (nhãn duy nhất): chọn gốc, chia các nhãn còn lại cho cây trái/phải theo mọi cách. Lấy mọi tập con của `{0,1,2,3}` ta được `485` cây (gồm cây rỗng). Đầu tiên ta chứng minh `reachable` (BFS phép lật) = `variants` (công thức đệ quy) trên mọi cây trong tập; rồi kiểm mọi cài đặt trên **mọi** `485² = 235 225` cặp.

In [22]:
def all_trees(labels):
    labels = tuple(labels)
    if not labels:
        return [None]
    out = []
    for i, root in enumerate(labels):
        rest = labels[:i] + labels[i + 1:]
        for mask in range(1 << len(rest)):
            left = tuple(rest[j] for j in range(len(rest)) if mask >> j & 1)
            right = tuple(rest[j] for j in range(len(rest)) if not mask >> j & 1)
            for lt in all_trees(left):
                for rt in all_trees(right):
                    out.append((root, lt, rt))
    return out


def from_tuple(t):
    return None if t is None else TreeNode(t[0], from_tuple(t[1]), from_tuple(t[2]))


pool = []
for k in range(0, 5):
    for labels in itertools.combinations(range(4), k):
        pool.extend(all_trees(labels))
print("số cây (nhãn ⊂ {0,1,2,3}, ≤ 4 nút):", len(pool), "| phân biệt:", len(set(pool)))
assert len(pool) == len(set(pool)) == 485

t0 = time.perf_counter()
for t in pool:
    assert reachable(t) == frozenset(variants(t)), t          # BFS phép lật = công thức đệ quy
print("reachable (BFS các phép lật) = variants (công thức) trên", len(pool), "cây;", round(time.perf_counter() - t0, 2), "s")

số cây (nhãn ⊂ {0,1,2,3}, ≤ 4 nút): 485 | phân biệt: 485
reachable (BFS các phép lật) = variants (công thức) trên 485 cây; 0.01 s


In [23]:
t0 = time.perf_counter()
trees = [from_tuple(t) for t in pool]
main_impls = ["recursion", "canon", "iter", "map", "Solution"]
true_pairs = 0
for i, tx in enumerate(pool):
    reach = reachable(tx)
    ax = trees[i]
    for j, ty in enumerate(pool):
        want = ty in reach                                       # oracle theo định nghĩa
        true_pairs += want
        by = trees[j]
        for name in main_impls:
            assert IMPLS[name](ax, by) == want, (name, tx, ty, want)
print("235 225 cặp:", len(main_impls), "cách khớp oracle; cặp tương đương lật:", true_pairs, ";", round(time.perf_counter() - t0, 1), "s")
assert len(pool) ** 2 == 235225

235 225 cặp: 5 cách khớp oracle; cặp tương đương lật: 2597 ; 2.0 s


In [24]:
# các cài đặt còn lại (brute và các class nộp) trên một mẫu cặp cố định
rng = random.Random(95101)
others = ["brute", "SolutionRecursion", "SolutionCanon", "SolutionIter", "SolutionMap", "SolutionBrute"]
t0 = time.perf_counter()
for _ in range(20000):
    i, j = rng.randrange(len(pool)), rng.randrange(len(pool))
    want = pool[j] in reachable(pool[i])
    if rng.random() < 0.4:                                      # lấy thêm cặp chắc chắn tương đương
        j = rng.randrange(len(pool))
        cand = sorted(reachable(pool[i]), key=str)
        pj = cand[rng.randrange(len(cand))]
        want, by = True, from_tuple(pj)
    else:
        by = trees[j]
    for name in others:
        assert IMPLS[name](trees[i], by) == want, (name, pool[i], want)
print("20 000 cặp mẫu:", len(others), "cài đặt còn lại khớp oracle;", round(time.perf_counter() - t0, 1), "s")

20 000 cặp mẫu: 6 cài đặt còn lại khớp oracle; 0.5 s


In [25]:
# 5 nút: mọi cây 5 nút nhãn {0..4} (5040 cây); với mẫu X: mọi Y thuộc lớp tương đương phải là True, vài Y ngẫu nhiên khác so oracle
rng = random.Random(95102)
big_pool = all_trees(range(5))
assert len(big_pool) == len(set(big_pool)) == 42 * 120
t0 = time.perf_counter()
classes = 0
for tx in rng.sample(big_pool, 400):
    reach = reachable(tx)
    ax = from_tuple(tx)
    for ty in list(reach) + rng.sample(big_pool, 12):
        want = ty in reach
        by = from_tuple(ty)
        for name in ("recursion", "canon", "iter", "map", "Solution", "brute"):
            assert IMPLS[name](ax, by) == want, (name, tx, ty)
    classes += 1
print("n = 5: 400 cây mẫu, mọi cây trong lớp tương đương + 12 cây ngẫu nhiên:", classes, "mẫu khớp;", round(time.perf_counter() - t0, 1), "s")

n = 5: 400 cây mẫu, mọi cây trong lớp tương đương + 12 cây ngẫu nhiên: 400 mẫu khớp; 0.3 s


### E. Tính chất (cây tới 100 nút, seed cố định)

(1) Bản sao lật ngẫu nhiên của cây luôn tương đương lật (đúng theo định nghĩa); (2) phản xạ: `X ~ X`; (3) đối xứng: `X ~ Y ⇔ Y ~ X`; (4) bắc cầu qua nhiều lần lật; (5) đổi **một** giá trị thành giá trị chưa dùng thì **không** còn tương đương; (6) bốn cách khớp nhau trên cặp ngẫu nhiên (kể cả cặp "gần đúng" hoán đổi hai giá trị); (7) với cây ≤ 9 nút khớp oracle `reachable`; (8) số lời gọi đệ quy `≤ 4n + 1`.

In [26]:
def count_calls(a, b):
    calls = [0]

    def rec(x, y):
        calls[0] += 1
        if x is None and y is None:
            return True
        if x is None or y is None or x.val != y.val:
            return False
        return (rec(x.left, y.left) and rec(x.right, y.right)) or (rec(x.left, y.right) and rec(x.right, y.left))

    return rec(a, b), calls[0]


def swap_two_values(root, rng):
    """Bản sao với hai giá trị bị hoán đổi (cây vẫn có giá trị duy nhất)."""
    cp = from_tuple(tup(root))
    nodes, stack = [], [cp]
    while stack:
        n = stack.pop()
        if n:
            nodes.append(n)
            stack.extend([n.left, n.right])
    if len(nodes) >= 2:
        u, v = rng.sample(nodes, 2)
        u.val, v.val = v.val, u.val
    return cp


def change_one_value(root, rng):
    """Bản sao đổi một giá trị thành giá trị chưa dùng trong [0, 99] (nếu còn)."""
    cp = from_tuple(tup(root))
    nodes, stack = [], [cp]
    while stack:
        n = stack.pop()
        if n:
            nodes.append(n)
            stack.extend([n.left, n.right])
    used = {n.val for n in nodes}
    free = [v for v in range(100) if v not in used]
    if not nodes or not free:
        return None
    rng.choice(nodes).val = rng.choice(free)
    return cp


rng = random.Random(20240951)
t0 = time.perf_counter()
n_false_swap = 0
for it in range(400):
    n = rng.randint(0, 100)
    x = random_tree(rng, n)
    y = copy_flip(x, rng, rng.choice([0.0, 0.3, 0.5, 1.0]))
    for name in ("recursion", "canon", "iter", "map", "Solution"):
        assert IMPLS[name](x, y), (name, it)                                  # (1)
        assert IMPLS[name](x, x) and IMPLS[name](y, x)                        # (2) (3)
    z = copy_flip(y, rng)
    assert flip_equiv(x, z) and flip_equiv(z, x)                              # (4)
    res, calls = count_calls(x, y)
    assert res and calls <= 4 * size(x) + 1, (calls, size(x))                 # (8)
    if n >= 1:
        w = change_one_value(x, rng)
        if w is not None:
            for name in ("recursion", "canon", "iter", "map"):
                assert not IMPLS[name](x, w), (name, it)                      # (5)
    if n >= 2:
        sw = swap_two_values(x, rng)
        answers = {name: IMPLS[name](x, sw) for name in ("recursion", "canon", "iter", "map")}
        assert len(set(answers.values())) == 1, answers                       # (6)
        n_false_swap += not answers["recursion"]
        if n <= 9:
            assert answers["recursion"] == oracle(x, sw)                      # (7)
print("400 cặp ngẫu nhiên (n ≤ 100): 8 tính chất đúng; số cặp hoán đổi hai giá trị mà KHÔNG còn tương đương:", n_false_swap, ";", round(time.perf_counter() - t0, 1), "s")

400 cặp ngẫu nhiên (n ≤ 100): 8 tính chất đúng; số cặp hoán đổi hai giá trị mà KHÔNG còn tương đương: 387 ; 0.4 s


In [27]:
# cây nhỏ ngẫu nhiên (n <= 9) đối chiếu oracle reachable với mọi cài đặt, kể cả cặp tuỳ ý (không cố ý tương đương)
rng = random.Random(95103)
t0 = time.perf_counter()
agree_true = 0
for _ in range(1500):
    n1, n2 = rng.randint(0, 8), rng.randint(0, 8)
    x = random_tree(rng, n1, rng.sample(range(10), max(n1, 1)) if n1 else None)
    y = random_tree(rng, n2, rng.sample(range(10), max(n2, 1)) if n2 else None) if rng.random() < 0.5 else copy_flip(x, rng)
    want = oracle(x, y)
    agree_true += want
    for name in IMPLS:
        assert IMPLS[name](x, y) == want, (name, tup(x), tup(y))
print("1500 cặp ngẫu nhiên n ≤ 8:", len(IMPLS), "cài đặt khớp oracle; số cặp True:", agree_true, ";", round(time.perf_counter() - t0, 1), "s")

1500 cặp ngẫu nhiên n ≤ 8: 11 cài đặt khớp oracle; số cặp True: 756 ; 0.2 s


### F. Giới hạn

`n = 100` nút, giá trị trong `[0, 99]`: chuỗi trái vs chuỗi phải (độ sâu 100, mọi nút đều phải lật), cây đầy đủ, cây ngẫu nhiên; trường hợp **không** tương đương mà chỉ khác ở một lá sâu nhất; và đo số lời gọi.

In [28]:
def chain(values, side):
    root = None
    for v in reversed(values):
        root = TreeNode(v, root, None) if side == "L" else TreeNode(v, None, root)
    return root


def complete(values):
    """Cây đầy đủ theo thứ tự tầng từ danh sách giá trị."""
    return build_tree(list(values))


vals = list(range(100))
cl, cr = chain(vals, "L"), chain(vals, "R")
assert size(cl) == size(cr) == 100
for name in ("recursion", "canon", "iter", "map", "Solution"):
    assert IMPLS[name](cl, cr), name                              # mọi nút phải lật
    assert not IMPLS[name](cl, chain(vals[:-1], "R")), name                      # thiếu một nút
cc = complete(range(100))
assert size(cc) == 100
cc_flipped = copy_flip(cc, random.Random(1), 1.0)
rng = random.Random(95104)
t0 = time.perf_counter()
for name in ("recursion", "canon", "iter", "map", "Solution"):
    assert IMPLS[name](cc, cc_flipped), name
# khác một lá sâu nhất: đổi giá trị lá đó thành 100 (ngoài [0, 99], chỉ để thử) => không còn tương đương
deep = from_tuple(tup(cc))
node = deep
while node.left or node.right:
    node = node.left or node.right
node.val = 100
for name in ("recursion", "canon", "iter", "map", "Solution"):
    assert not IMPLS[name](cc, deep), name
    assert not IMPLS[name](deep, cc_flipped), name
res, calls = count_calls(cl, cr)
print("chuỗi 100 nút lật hết:", res, "| số lời gọi:", calls, "<= 4n+1 =", 4 * 100 + 1)
assert calls <= 401
for _ in range(100):
    x = random_tree(rng, 100)
    y = copy_flip(x, rng)
    assert flip_equiv(x, y) and flip_equiv_canon(x, y) and flip_equiv_iter(x, y) and flip_equiv_map(x, y)
print("n = 100: chuỗi, cây đầy đủ, 100 cây ngẫu nhiên đều đúng;", round(time.perf_counter() - t0, 2), "s")

chuỗi 100 nút lật hết: True | số lời gọi: 300 <= 4n+1 = 401


n = 100: chuỗi, cây đầy đủ, 100 cây ngẫu nhiên đều đúng; 0.03 s


## 13. Các lỗi người mới hay gặp

### Lỗi 1: chỉ so sánh thẳng (quên lật)

In [29]:
a, b = build_tree([1, 2, 3]), build_tree([1, 3, 2])
print("bug:", bug_no_flip(a, b), "| đúng:", Solution().flipEquiv(a, b))
assert bug_no_flip(a, b) is False and Solution().flipEquiv(a, b) is True

bug: False | đúng: True


**Vì sao sai:** đề cho phép lật, cần thử thêm nhánh chéo. **Đúng:** `(thẳng) or (chéo)`.

### Lỗi 2: dùng `and` thay `or` giữa hai nhánh

In [30]:
print("bug:", bug_and_instead_of_or(r1, r2), "| đúng:", Solution().flipEquiv(r1, r2))
assert bug_and_instead_of_or(r1, r2) is False and Solution().flipEquiv(r1, r2) is True

bug: False | đúng: True


**Vì sao sai:** chỉ cần **một** trong hai cách ghép thành công (lật hoặc không lật) — không phải cả hai.

### Lỗi 3: trả `True` khi một bên rỗng

In [31]:
print("bug:", bug_none_is_true(None, build_tree([1])), "| đúng:", Solution().flipEquiv(None, build_tree([1])))
assert bug_none_is_true(None, build_tree([1])) is True and Solution().flipEquiv(None, build_tree([1])) is False

bug: True | đúng: False


**Vì sao sai:** "một rỗng một không rỗng" nghĩa là hình dạng khác nhau ⇒ `False`. Chỉ khi **cả hai** rỗng mới `True`.

### Lỗi 4: quên so sánh giá trị gốc

In [32]:
a, b = build_tree([1, 2, 3]), build_tree([1, 2, 4])
print("bug:", bug_no_root_check(a, b), "| đúng:", Solution().flipEquiv(a, b))
a2, b2 = build_tree([5]), build_tree([6])
assert bug_no_root_check(a2, b2) is True and Solution().flipEquiv(a2, b2) is False

bug: True | đúng: False


**Vì sao sai:** phép lật không đổi giá trị; hai cây có gốc khác nhau không bao giờ tương đương. **Đúng:** `root1.val != root2.val ⇒ False`.

### Lỗi 5: so sánh theo tập giá trị / duyệt thay vì cấu trúc

In [33]:
a, b = build_tree([1, 2, 3, 4]), build_tree([1, 2, 3, None, None, 4])      # 4 là con của 2 hay của 3?
print("bug:", bug_value_multiset(a, b), "| đúng:", Solution().flipEquiv(a, b))
assert bug_value_multiset(a, b) is True and Solution().flipEquiv(a, b) is False

bug: True | đúng: False


**Vì sao sai:** cùng tập giá trị chưa chắc cùng cấu trúc: `4` là con của `2` hay của `3`? **Đúng:** so sánh cấu trúc cha–con, không chỉ giá trị.

### Lỗi 6 (bonus): truy cập `.val` của `None`

In [34]:
a, b = build_tree([1, None, 2]), build_tree([1, None, 2])
try:
    bug_attribute_error(a, b)
    print("không lỗi?")
except AttributeError as e:
    print("AttributeError:", e)
assert Solution().flipEquiv(a, b) is True

AttributeError: 'NoneType' object has no attribute 'val'


**Vì sao sai:** `a.left` có thể là `None`; phải kiểm tra `None` (hoặc dùng hàm `lv`) trước khi đọc `.val`.

## 14. Mẹo & nhận dạng pattern

**Nhận dạng:** "hai cây giống nhau sau các phép biến đổi cục bộ giao hoán (đổi chỗ hai con)" → đệ quy **hai nhánh** (`thẳng or chéo`), hoặc đưa về **dạng chuẩn tắc** (sắp xếp các con).

**Mẹo:**

1. Điều kiện dừng đúng thứ tự: cả hai `None` → `True`; một `None` hoặc giá trị khác → `False`.
2. Với giá trị duy nhất, nhánh sai tự bị loại ngay ở kiểm tra giá trị ⇒ đệ quy `O(n)`.
3. Dạng chuẩn tắc (sắp xếp các con) đúng cả khi giá trị trùng.
4. Dùng bộ lồng nhau `(val, trái, phải)` để băm/so sánh cây và để viết oracle.
5. Oracle theo **định nghĩa nguyên văn** (BFS các phép lật) trên cây nhỏ, vét cạn mọi cây ≤ 4 nút.
6. Tự viết `build_tree`/`to_level` kiểu LeetCode: nhớ rằng **con của nút trống không xuất hiện** trong danh sách.

**Họ bài cùng pattern:** Same Tree (100), Symmetric Tree (101), Invert Binary Tree (226), Subtree of Another Tree (572), Leaf-Similar Trees (872), Find Duplicate Subtrees (652), Tree Isomorphism.

## 15. Biến thể

1. **Giá trị có thể trùng:** đệ quy hai nhánh và dạng chuẩn tắc vẫn đúng; ghép cặp theo giá trị (Cách 4) và ánh xạ cha→con (Cách 5) có thể **sai**.
2. **Số cây phân biệt tương đương lật với `X`:** `2^(số nút có ít nhất một con)` (giá trị duy nhất).
3. **Số phép lật tối thiểu** để biến `X` thành `Y` (hoặc `-1`).

Cài và đối chiếu với BFS các phép lật:

In [35]:
# --- Biến thể 1: giá trị trùng
dup1 = build_tree([1, 2, 2, 3, None, None, 4])          # 1(2(3,-), 2(-,4))
dup2 = build_tree([1, 2, 2, 4, None, None, 3])          # 1(2(4,-), 2(-,3))
t1, t2 = tup(dup1), tup(dup2)
truth_dup = t2 in reachable(t1)
print("giá trị trùng: đúng =", truth_dup,
      "| recursion:", flip_equiv(dup1, dup2), "| canon:", flip_equiv_canon(dup1, dup2),
      "| iter (giá trị duy nhất):", flip_equiv_iter(dup1, dup2), "| map:", flip_equiv_map(dup1, dup2))
assert flip_equiv(dup1, dup2) == truth_dup == flip_equiv_canon(dup1, dup2)
assert flip_equiv_map(dup1, dup2) != truth_dup            # ánh xạ cha → tập con bị ghi đè khi trùng giá trị


# kiểm trên mọi cây ≤ 4 nút với nhãn từ {0, 1} (nhiều giá trị trùng)
def all_trees_dup(n, alphabet):
    if n == 0:
        return [None]
    out = []
    for root in alphabet:
        for k in range(n):
            for lt in all_trees_dup(k, alphabet):
                for rt in all_trees_dup(n - 1 - k, alphabet):
                    out.append((root, lt, rt))
    return out


dup_pool = []
for n in range(0, 5):
    dup_pool.extend(all_trees_dup(n, (0, 1)))
dup_pool = list(dict.fromkeys(dup_pool))
bad_iter = bad_map = 0
for tx in dup_pool:
    reach = reachable(tx)
    ax = from_tuple(tx)
    for ty in dup_pool:
        want = ty in reach
        by = from_tuple(ty)
        assert flip_equiv(ax, by) == want and flip_equiv_canon(ax, by) == want
        bad_iter += flip_equiv_iter(ax, by) != want
        bad_map += flip_equiv_map(ax, by) != want
print(f"{len(dup_pool)} cây nhãn {{0,1}} (cho phép trùng): recursion và canon luôn đúng; iter sai {bad_iter} cặp, map sai {bad_map} cặp")
assert bad_iter > 0 and bad_map > 0

giá trị trùng: đúng = True | recursion: True | canon: True | iter (giá trị duy nhất): False | map: False


275 cây nhãn {0,1} (cho phép trùng): recursion và canon luôn đúng; iter sai 64 cặp, map sai 9556 cặp


In [36]:
# --- Biến thể 2: số cây phân biệt tương đương lật = 2^(số nút có ít nhất một con)
def internal_count(t):
    if t is None:
        return 0
    v, l, r = t
    return (1 if (l is not None or r is not None) else 0) + internal_count(l) + internal_count(r)


for t in pool:
    assert len(reachable(t)) == 2 ** internal_count(t), t
print("số cây tương đương lật = 2^(nút có con) đúng trên", len(pool), "cây")


# --- Biến thể 3: số phép lật tối thiểu
def min_flips(a, b):
    """Số phép lật tối thiểu để biến a thành b; -1 nếu không thể."""
    if a is None and b is None:
        return 0
    if a is None or b is None or a.val != b.val:
        return -1
    best = -1
    s1, s2 = min_flips(a.left, b.left), min_flips(a.right, b.right)
    if s1 >= 0 and s2 >= 0:
        best = s1 + s2
    c1, c2 = min_flips(a.left, b.right), min_flips(a.right, b.left)
    if c1 >= 0 and c2 >= 0 and (best < 0 or 1 + c1 + c2 < best):
        best = 1 + c1 + c2
    return best


def bfs_dist(t):
    dist = {t: 0}
    queue = deque([t])
    while queue:
        cur = queue.popleft()
        for nxt in one_flips(cur):
            if nxt not in dist:
                dist[nxt] = dist[cur] + 1
                queue.append(nxt)
    return dist


t0 = time.perf_counter()
for tx in pool:
    dist = bfs_dist(tx)
    ax = from_tuple(tx)
    for ty in pool:
        assert min_flips(ax, from_tuple(ty)) == dist.get(ty, -1), (tx, ty)
print("số phép lật tối thiểu khớp BFS trên", len(pool) ** 2, "cặp;", round(time.perf_counter() - t0, 1), "s")

số cây tương đương lật = 2^(nút có con) đúng trên 485 cây


số phép lật tối thiểu khớp BFS trên 235225 cặp; 0.2 s


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Giá trị trùng? | Khi nào dùng |
|---|---|---|---|---|---|
| 1. Liệt kê mọi cách lật | tập `2^k` cây | `O(n·2^k)` | `O(n·2^k)` | đúng | oracle cây nhỏ |
| 2. Đệ quy hai nhánh | thẳng `or` chéo | `O(n)` (duy nhất) | `O(h)` | đúng | **đáp án chính** |
| 3. Dạng chuẩn tắc | sắp xếp hai con | `O(n·h)` | `O(n)` | đúng | tổng quát, so nhiều cây |
| 4. Duyệt lặp ghép cặp | ghép theo giá trị | `O(n)` | `O(n)` | **sai** | cần tránh đệ quy |
| 5. Ánh xạ cha → con | quan hệ cha–con | `O(n)` | `O(n)` | **sai** | so sánh nhanh |

Mọi cách khớp oracle (BFS theo định nghĩa) trên **mọi** `235 225` cặp cây ≤ 4 nút (mục 12D). Trong phần 15 ta thấy hai cách cuối sai khi giá trị trùng.

## 17. Bài tập vận dụng

Các cell TODO **chưa được gọi**: tự điền rồi bỏ comment dòng test cuối cell.

### Bài 1 (đoán output)

Đoán trước rồi chạy:

In [37]:
print(Solution().flipEquiv(build_tree([1, 2, 3]), build_tree([1, 3, 2])),
      Solution().flipEquiv(build_tree([1, 2]), build_tree([1, None, 2])),
      Solution().flipEquiv(build_tree([1, 2, 3]), build_tree([1, 2, 4])),
      Solution().flipEquiv(build_tree([]), build_tree([0])))

True True False False


### Bài 2 (tính tay)

`X = [1,2,3,4]` (gốc `1`, con trái `2` có con trái `4`, con phải `3`), `Y = [1,3,2,null,null,4]`, `Z = [1,3,2,null,null,null,4]`. (a) `X ~ Y`? Cần mấy phép lật? (b) `X ~ Z`? Cần mấy phép lật? Kiểm tra ở mục 19.

### Bài 3 (điền TODO)

Hoàn thiện đệ quy hai nhánh:

In [38]:
def flip_equiv_todo(a, b):
    if a is None and b is None:
        return True
    if a is None or b is None or a.val != b.val:
        return False
    pass            # TODO: trả về (thẳng) or (chéo)


# Khi làm xong, bỏ comment để kiểm tra:
# assert flip_equiv_todo(r1, r2) and not flip_equiv_todo(None, build_tree([1]))

### Bài 4 (tự viết lại)

Không nhìn lời giải, viết `class MySolution` bằng **dạng chuẩn tắc** (sắp xếp hai con):

In [39]:
class MySolution:
    def flipEquiv(self, root1, root2):
        pass            # TODO


# Khi làm xong:
# assert MySolution().flipEquiv(r1, r2) and not MySolution().flipEquiv(None, build_tree([1]))

### Bài 5 (mini challenge): số phép lật tối thiểu

Viết `min_flips_ans(a, b)` trả về số phép lật tối thiểu để biến `a` thành `b` (hoặc `-1` nếu không thể). Gợi ý: ở mỗi nút so cách ghép thẳng (không tốn phép lật tại nút) và cách ghép chéo (tốn 1), lấy nhỏ hơn trong các cách hợp lệ. Kiểm bằng `bfs_dist` ở mục 15.

In [40]:
def min_flips_ans(a, b):
    pass            # TODO


# Khi làm xong:
# assert min_flips_ans(r1, r2) == 3

## 18. Gợi ý 3 mức

**Bài 1**

- Mức 1: lật tại một nút chỉ đổi chỗ hai con.
- Mức 2: `[1,2]` là `1` có con trái `2`; `[1,null,2]` là `1` có con phải `2`: lật được.
- Mức 3: kết quả `True True False False`.

**Bài 2**

- Mức 1: viết `X = 1(2(4,·), 3)`.
- Mức 2: lật gốc: `1(3, 2(4,·))`; có bằng `Y`? Có bằng `Z` không (so vị trí của `4`)?
- Mức 3: `X ~ Y` với 1 phép lật (tại gốc); `X ~ Z` với 2 phép lật (tại gốc và tại `2`).

**Bài 3**

- Mức 1: hai nhánh, mỗi nhánh gồm hai lời gọi đệ quy nối bằng `and`.
- Mức 2: thẳng: `(a.left, b.left)`, `(a.right, b.right)`; chéo: `(a.left, b.right)`, `(a.right, b.left)`.
- Mức 3: `return (A and B) or (C and D)`.

**Bài 4**

- Mức 1: `canon(None) = ()`.
- Mức 2: `canon(n) = (n.val, *sorted([canon(n.left), canon(n.right)]))`.
- Mức 3: so sánh `canon(root1) == canon(root2)`.

**Bài 5**

- Mức 1: giống đệ quy nhưng trả về số thay vì bool; `-1` nghĩa là không thể.
- Mức 2: thẳng: `f(al,bl)+f(ar,br)`; chéo: `1+f(al,br)+f(ar,bl)`; bỏ cách có số hạng `-1`.
- Mức 3: lấy `min` các cách hợp lệ; nếu không có cách nào thì `-1`.

## 19. Đáp án tham khảo

Mỗi đáp án đều có `assert`.

### Bài 1

In [41]:
vals = (Solution().flipEquiv(build_tree([1, 2, 3]), build_tree([1, 3, 2])),
        Solution().flipEquiv(build_tree([1, 2]), build_tree([1, None, 2])),
        Solution().flipEquiv(build_tree([1, 2, 3]), build_tree([1, 2, 4])),
        Solution().flipEquiv(build_tree([]), build_tree([0])))
print(vals)
assert vals == (True, True, False, False)
print("Bài 1 OK")

(True, True, False, False)
Bài 1 OK


### Bài 2

In [42]:
X, Y, Z = build_tree([1, 2, 3, 4]), build_tree([1, 3, 2, None, None, 4]), build_tree([1, 3, 2, None, None, None, 4])
print(tup(X), tup(Y), tup(Z), sep="\n")
assert Solution().flipEquiv(X, Y) and Solution().flipEquiv(X, Z)
assert min_flips(X, Y) == 1 and min_flips(X, Z) == 2
assert bfs_dist(tup(X))[tup(Y)] == 1 and bfs_dist(tup(X))[tup(Z)] == 2
print("Bài 2 OK: X~Y cần 1 phép lật, X~Z cần 2 phép lật")

(1, (2, (4, None, None), None), (3, None, None))
(1, (3, None, None), (2, (4, None, None), None))
(1, (3, None, None), (2, None, (4, None, None)))
Bài 2 OK: X~Y cần 1 phép lật, X~Z cần 2 phép lật


### Bài 3

In [43]:
def flip_equiv_answer(a, b):
    if a is None and b is None:
        return True
    if a is None or b is None or a.val != b.val:
        return False
    return (flip_equiv_answer(a.left, b.left) and flip_equiv_answer(a.right, b.right)) or \
           (flip_equiv_answer(a.left, b.right) and flip_equiv_answer(a.right, b.left))


assert flip_equiv_answer(r1, r2) and not flip_equiv_answer(None, build_tree([1]))
for ta, tb in itertools.islice(itertools.product(pool, repeat=2), 0, 235225, 97):
    assert flip_equiv_answer(from_tuple(ta), from_tuple(tb)) == (tb in reachable(ta))
print("Bài 3 OK")

Bài 3 OK


### Bài 4

In [44]:
class MySolutionAnswer:
    def flipEquiv(self, root1, root2):
        def canon(n):
            if n is None:
                return ()
            return (n.val, *sorted([canon(n.left), canon(n.right)]))

        return canon(root1) == canon(root2)


assert MySolutionAnswer().flipEquiv(r1, r2) and not MySolutionAnswer().flipEquiv(None, build_tree([1]))
for ta, tb in itertools.islice(itertools.product(pool, repeat=2), 0, 235225, 97):
    assert MySolutionAnswer().flipEquiv(from_tuple(ta), from_tuple(tb)) == (tb in reachable(ta))
print("Bài 4 OK")

Bài 4 OK


### Bài 5

In [45]:
def min_flips_answer(a, b):
    if a is None and b is None:
        return 0
    if a is None or b is None or a.val != b.val:
        return -1
    options = []
    s1, s2 = min_flips_answer(a.left, b.left), min_flips_answer(a.right, b.right)
    if s1 >= 0 and s2 >= 0:
        options.append(s1 + s2)
    c1, c2 = min_flips_answer(a.left, b.right), min_flips_answer(a.right, b.left)
    if c1 >= 0 and c2 >= 0:
        options.append(1 + c1 + c2)
    return min(options) if options else -1


assert min_flips_answer(r1, r2) == 3
for tx in pool[::5]:
    dist = bfs_dist(tx)
    for ty in pool[::3]:
        assert min_flips_answer(from_tuple(tx), from_tuple(ty)) == dist.get(ty, -1)
print("Bài 5 OK: ví dụ 1 cần 3 phép lật (tại 1, 3, 5)")

Bài 5 OK: ví dụ 1 cần 3 phép lật (tại 1, 3, 5)


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template (hai cây giống nhau sau các phép đổi chỗ con → đệ quy hai nhánh thẳng/chéo):**

```python
class Solution:
    def flipEquiv(self, root1: Optional[TreeNode], root2: Optional[TreeNode]) -> bool:
        if root1 is None and root2 is None:
            return True
        if root1 is None or root2 is None or root1.val != root2.val:
            return False
        return (self.flipEquiv(root1.left, root2.left) and self.flipEquiv(root1.right, root2.right)) or \
               (self.flipEquiv(root1.left, root2.right) and self.flipEquiv(root1.right, root2.left))
```

**Checklist trước khi submit:**

- [ ] Hai cây rỗng → `True`; một rỗng một không → `False`?
- [ ] So **giá trị gốc** trước khi đệ quy?
- [ ] `(thẳng and thẳng) or (chéo and chéo)` — `and` trong mỗi nhánh, `or` giữa hai nhánh?
- [ ] Không truy cập `.val` của `None`?
- [ ] Hiểu vì sao giá trị duy nhất cho `O(n)` (số lời gọi `≤ 4n + 1`)?
- [ ] Đã thử ví dụ 1, `[]` vs `[]`, `[]` vs `[1]`, chuỗi trái vs chuỗi phải?
- [ ] Đúng tên `class Solution`, `flipEquiv(self, root1, root2)`, không in debug.